# 01 — Underlying Price Model

**Phase 14 of the Inflexion build** — derives the parameters of `params.json` that gate PARTIAL mode (see `spec.md` §9).

**This notebook (Task 14.2):** simulate the price of token0/USDC underlying a Uniswap v3 LP position.

Models we build, in order:
1. **Baseline GBM** (geometric Brownian motion) — calibration starting point
2. **Jump-diffusion** (Merton or Kou) — captures crypto's discontinuous moves
3. **Historical bootstrap** — resamples actual ETH / BTC / ARB returns
4. **Common-factor stress** — adds a market-wide crash factor for correlated scenarios (the core danger for the insurance fund)

**Next notebooks:**
- `02_position_structures.ipynb` — distributions of LP range widths × moneyness
- `03_path_to_il.ipynb` — path → IL via spec §3.1 formulas
- `04_portfolio_waterfall.ipynb` — MM-collateral-first / fund-covers-tail waterfall
- `05_stress.ipynb` — correlated crash, vol regime shift, utilization spike
- `06_calibrate_params.ipynb` — derive `params.json`

## Smoke test — environment is wired correctly

In [ ]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import inflexion_quant
from inflexion_quant import il

rng = np.random.default_rng(seed=42)
print(f"inflexion_quant {inflexion_quant.__version__}")
print(f"numpy {np.__version__} | scipy {stats.__name__.split('.')[0]} OK | matplotlib OK")
print(f"il module loaded: {il.__doc__.splitlines()[0]}")

In [ ]:
# Phase 14.2 will land the actual GBM + jump-diffusion + bootstrap code here.
# Smoke test for now: generate a single GBM path and plot it.

S0 = 3000.0      # initial ETH/USD
mu = 0.0         # drift (risk-neutral baseline)
sigma = 0.65     # annualized vol (matches spec.md §2.3 ETH example)
T = 30 / 365     # 30-day horizon
N = 24 * 30      # hourly steps
dt = T / N

z = rng.standard_normal(N)
log_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
path = S0 * np.exp(np.cumsum(log_returns))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.linspace(0, T, N), path, lw=1.2)
ax.set_xlabel("time (years)"); ax.set_ylabel("price")
ax.set_title(f"Single GBM smoke test — sigma={sigma:.0%}, T={int(T*365)}d")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---

**Phase 14.2 starts here** — replace the smoke test above with: (1) vectorised GBM for many paths at once, (2) Kou jump-diffusion (asymmetric exponential jumps — captures the up/down asymmetry crypto exhibits), (3) historical bootstrap from a real ETH/USDC dataset (Coingecko or Binance daily), (4) a common-factor stress version that adds a correlated crash component across markets.